In [2]:
import os
import pandas as pd
import nltk
import re
import torch
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from transformers import AutoTokenizer, T5ForConditionalGeneration

c:\Users\yes\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yes\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yes\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
path = 'ims'
years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

In [5]:
files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f)) and 'annotation' not in f.lower()]
print(files)

['Belova_IMS_2014_rus.txt', 'Belov_IMS_2018_rus.txt', 'Boyarsky_IMS_2015_rus.txt', 'Boyarsky_IMS_2019_rus.txt', 'ChizhikMelnikovaZakharov_IMS_2022.txt', 'Chizhik_IMS_2021_rus.txt', 'Dubovik_IMS_2017_rus.txt', 'Gavrilkina_IMS_2023.txt', 'Godgildieva_IMS_2016_rus.txt', 'Ivanchenko_IMS_2013_rus.txt', 'Khokhlova_IMS_2020_rus.txt', 'MitrofanovaGolubev_IMS_2024_rus.txt']


In [6]:
texts = []

In [7]:
for file in files:
    with open(os.path.join(path, file), 'r', encoding='utf-8') as f:
        text = f.read().strip()
       
        # фильтрация метаинформации
        lines = text.split('\n')
        filtered_text = []
        skip = True
        for line in lines:
            if line.strip().lower() == 'введение':  # начало основного текста
                skip = False
            if not skip and line.strip():
                filtered_text.append(line)
        filtered_text = ' '.join(filtered_text)

        # очистка от лишних символов
        filtered_text = re.sub(r'[•]', '', filtered_text)
        name = file[:-4]
        year_match = re.search(r'(20\d{2})', file)
        year = year_match.group(0) if year_match else None
        if year and int(year) in years and filtered_text:
            texts.append((name, filtered_text, year))

In [8]:
df = pd.DataFrame(texts, columns=['names', 'texts', 'years'])
print(df)

                               names  \
0                Belova_IMS_2014_rus   
1              Boyarsky_IMS_2019_rus   
2  ChizhikMelnikovaZakharov_IMS_2022   
3             Khokhlova_IMS_2020_rus   

                                               texts years  
0  Введение В настоящее время интернет дает огром...  2014  
1  Введение Средства массовой информации каждый д...  2019  
2  Введение Одним из важнейших аспектов социологи...  2022  
3  Введение Исследование сочетаемости не теряет с...  2020  


In [9]:
summarizer = LexRankSummarizer()

In [10]:
annotations_sumy = []

In [11]:
for idx, (name, text, year) in enumerate(zip(df.names, df.texts, df.years), 1):
    # дополнительная очистка текста перед токенизацией
    text = re.sub(r'\s+', ' ', text).strip()
    parser = PlaintextParser.from_string(text, Tokenizer('russian'))
    summary = summarizer(parser.document, 3)  # 3 предложения
    abstract = [str(sentence).strip() for sentence in summary if str(sentence).strip()]
    annotation = ' '.join(abstract)
    annotations_sumy.append(annotation)
    print(f"Статья {idx}: {name} ({year})\n{annotation}\n")
print(f"Обработано {len(annotations_sumy)} аннотаций LexRank")

Статья 1: Belova_IMS_2014_rus (2014)
В настоящее время регионы России реализуют различные программы и проекты, призванные привлечь граждан к получению государственных и муниципальных услуг в электронной форме и направленные на повышение базовой компьютерной грамотности, осведомленности граждан об инновационных механизмах получения услуг с использованием ИТ. Далее на примере опыта Нижегородской области по реализации социального проекта «Электронный гражданин Нижегородской области» будет рассмотрено применение различных инструментов популяризации электронного правительства. Проект «Электронный гражданин Нижегородской области» - мультикомпонентный проект, который сочетает в себе:  очные курсы компьютерной грамотности;  ресурсы для самостоятельного обучения граждан компьютерной грамотности;  инструменты для популяризации ресурсов Электронного правительства Нижегородской области. Для этого необходимо использовать рекламные сообщения, содержащие слова и понятия «быстро, без очередей, удоб

In [12]:
annots_sumy = pd.DataFrame(annotations_sumy, columns=['annotations_sumy'])
sumy = pd.concat([df, annots_sumy], axis=1)
print("\nDataFrame с аннотациями LexRank:")
print(sumy)


DataFrame с аннотациями LexRank:
                               names  \
0                Belova_IMS_2014_rus   
1              Boyarsky_IMS_2019_rus   
2  ChizhikMelnikovaZakharov_IMS_2022   
3             Khokhlova_IMS_2020_rus   

                                               texts years  \
0  Введение В настоящее время интернет дает огром...  2014   
1  Введение Средства массовой информации каждый д...  2019   
2  Введение Одним из важнейших аспектов социологи...  2022   
3  Введение Исследование сочетаемости не теряет с...  2020   

                                    annotations_sumy  
0  В настоящее время регионы России реализуют раз...  
1  В некоторых случаях удается уточнить класс при...  
2  Значит, для оценки ситуации в стране следует о...  
3  Обзор методов Традиционные методы извлечения л...  


In [13]:
# инициализация T5 модели
model_name = "IlyaGusev/rut5_base_sum_gazeta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [14]:
annotations_t5 = []

In [15]:
for idx, (name, text, year) in enumerate(zip(df.names, df.texts, df.years), 1):
    article_text = text
    input_ids = tokenizer([article_text], max_length=600, add_special_tokens=True, padding="max_length", truncation=True, return_tensors="pt")["input_ids"]
    output_ids = model.generate(input_ids=input_ids, min_length=50, max_length=250, no_repeat_ngram_size=4)[0]
    summary = tokenizer.decode(output_ids, skip_special_tokens=True)
    annotations_t5.append(summary)
    print(f"Статья {idx}: {name} ({year})\n{summary}\n")
print(f"Обработано {len(annotations_t5)} аннотаций T5")

Статья 1: Belova_IMS_2014_rus (2014)
Государственная власть все активнее выходит в интернет-пространство через ресурсы, предполагающие прямое взаимодействие с аудиторией и повышение готовности граждан к получению государственных и муниципальных услуг. Эксперты считают, что интернет дает огромные возможности для продвижения социальных проектов и инновационных проектов.

Статья 2: Boyarsky_IMS_2019_rus (2019)
Выделение именованных сущностей – одна из ключевых задач извлечения информации из неструктурированных или слабоструктурированных документов, а также выделение из текстов названий географических объектов, к которым может иметь отношение данное сообщение.

Статья 3: ChizhikMelnikovaZakharov_IMS_2022 (2022)
Социальное настроение является доминантой функционирующего сознания, которое является индикатором определения уровня благополучия и социальной устроенности общества. Однако для оценки ситуации в стране следует обращаться к анализу социального настроения, который формируется на основ

In [16]:
annots_t5 = pd.DataFrame(annotations_t5, columns=['annotations_t5'])
full = pd.concat([sumy, annots_t5], axis=1)
print("\nПолный DataFrame:")
print(full)


Полный DataFrame:
                               names  \
0                Belova_IMS_2014_rus   
1              Boyarsky_IMS_2019_rus   
2  ChizhikMelnikovaZakharov_IMS_2022   
3             Khokhlova_IMS_2020_rus   

                                               texts years  \
0  Введение В настоящее время интернет дает огром...  2014   
1  Введение Средства массовой информации каждый д...  2019   
2  Введение Одним из важнейших аспектов социологи...  2022   
3  Введение Исследование сочетаемости не теряет с...  2020   

                                    annotations_sumy  \
0  В настоящее время регионы России реализуют раз...   
1  В некоторых случаях удается уточнить класс при...   
2  Значит, для оценки ситуации в стране следует о...   
3  Обзор методов Традиционные методы извлечения л...   

                                      annotations_t5  
0  Государственная власть все активнее выходит в ...  
1  Выделение именованных сущностей – одна из ключ...  
2  Социальное настроени

In [18]:
result_df = full[['names', 'years', 'annotations_sumy', 'annotations_t5']]

In [20]:
result_df.to_csv('summarization.csv', index=False, encoding='utf-8')